# Chapter 2: Searching State Spaces

```{admonition} Learning Objectives
:class: tip
- Understand state space representation as graphs
- Master uninformed search algorithms: BFS, DFS, UCS, Iterative Deepening
- Master informed search algorithms: Greedy Best-First, A*
- Learn heuristic design and admissibility
- Apply local search methods: Hill Climbing, Simulated Annealing, Tabu Search
- Understand genetic algorithms for optimization
- Analyze algorithm complexity, optimality, and completeness
```

## 2.1 Introduction

Search is one of the most fundamental problem-solving techniques in artificial intelligence. Many AI problems can be formulated as finding a sequence of actions that transforms an **initial state** into a **goal state**.

### Core Concept

The goal of an agent is to:
- Start from an initial state
- Perform a sequence of actions
- Reach a goal state (or a state with high desirability)

### Two Main Scenarios

1. **Goal-Oriented Search**: Find a path from start to goal
   - Example: Solving a maze, solving a puzzle
   - Cost may be associated with paths

2. **Optimization Search**: Find state with minimal loss
   - Example: Eight-queens placement, scheduling
   - Cost is associated with states themselves

### Applications

```{mermaid}
graph TD
    A[Search Problems] --> B[Route Planning]
    A --> C[Puzzle Solving]
    A --> D[Game Playing]
    A --> E[Scheduling]
    A --> F[Robot Navigation]
    B --> B1[GPS Navigation]
    C --> C1[8-Puzzle, Rubik's Cube]
    D --> D1[Chess Move Selection]
    E --> E1[Resource Allocation]
    F --> F1[Path Planning]
    style A fill:#FFE4E1
```

### Why Search Matters

1. **No Direct Formula**: Many problems have no closed-form solution
2. **Large Solution Space**: Too many possibilities to enumerate
3. **Need Systematic Methods**: Avoid aimless wandering
4. **Trade-offs**: Different strategies balance time, memory, and solution quality

### 2.1.1 State Space as a Graph

Modeling the state space as a **directed graph** enables systematic search:

**Key Concepts:**

- **Node**: Represents a state
- **Edge**: Represents a transition (action) from one state to another
- **Path**: Sequence of edges from initial state to goal

```{mermaid}
graph LR
    S[Start State] -->|Action 1| A[State A]
    S -->|Action 2| B[State B]
    A -->|Action 3| C[State C]
    B -->|Action 4| C
    C -->|Action 5| G[Goal State]
    style S fill:#90EE90
    style G fill:#FFB6C1
```

### Example: Eight-Queens Problem

**Problem**: Place 8 queens on a chessboard so none attack each other.

**State Space Design:**

**Approach 1** (Incremental):  
- **State**: Placement of k ≤ 8 queens (no attacks)
- **Action**: Add one more queen
- **Initial State**: Empty board (k=0)
- **Goal State**: 8 queens placed (k=8)
- **Graph Type**: Directed Acyclic Graph (DAG)

**Approach 2** (Complete State):  
- **State**: Any placement of 8 queens (attacks allowed)
- **Action**: Move one queen to different square
- **Initial State**: Random placement
- **Goal State**: No attacks
- **Graph Type**: Undirected graph with cycles

```{note}
The choice of state representation affects:
- Size of state space
- Which search algorithm works best
- Efficiency of finding solutions
```

### State Space Size

| Problem | State Space Size |
|---------|------------------|
| 8-Queens (any placement) | 64C8 ≈ 4.4 × 10⁸ |
| 8-Puzzle | 9! = 362,880 |
| Chess (valid positions) | > 10⁴³ |
| Rubik's Cube | ≈ 4.3 × 10¹⁹ |

```{warning}
For large state spaces like chess, it's **impossible to store the entire graph**. Search algorithms must explore states dynamically!
```

### 2.1.2 Search Space vs Search Tree

The **state space graph** represents all possible states and transitions in the problem domain, while the **search tree** represents the specific paths taken during the search process.

**Key Differences:**

| Aspect | State Space Graph | Search Tree |
|--------|-------------------|-------------|
| **Definition** | All possible states and transitions | Paths explored by search algorithm |
| **Duplicates** | Each state appears once | Same state may appear multiple times |
| **Size** | Fixed by problem | Can be infinitely larger |
| **Cycles** | May contain cycles | Represents unrolled paths |

**Example: State Space Graph**
```{mermaid}
graph LR
    S((S)) --> a((a))
    S --> d((d))
    S --> p((p))
    a --> b((b))
    a --> c((c))
    d --> b
    d --> c
    d --> e((e))
    c --> e
    e --> f((f))
    f --> G((G))
    e --> h((h))
    e --> r((r))
    p --> q((q))
    q --> h
    p --> e
    style S fill:#90EE90
    style G fill:#FFB6C1
```

**Corresponding Search Tree:**

The search tree unfolds paths from the root, potentially duplicating states reached via different paths.

```mermaid
graph TD
		S:::highlight --> a
		S --> d
		S --> p

		a --> b
		a --> c

		d --> b2[b]
		d --> c2[c]
		d --> e1[e]

		c --> e2[e]

		p --> q
		p --> e3[e]

		q --> h1[h]

		e1 --> f1[f]
		e1 --> h2[h]
		e1 --> r1[r]

		e2 --> f2[f]
		e2 --> h3[h]
		e2 --> r2[r]

		e3 --> f3[f]
		e3 --> h4[h]
		e3 --> r3[r]

		f1 --> G1[G]:::highlight
		f2 --> G2[G]:::highlight
		f3 --> G3[G]:::highlight
		classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
```

```{warning}
The size of the search tree can be **infinitely larger** than the state space graph due to:
- Cycles creating infinite branches
- Multiple paths leading to the same state
- Each path becoming a separate branch in the tree
```

**Example with Cycles:**

Consider a graph: 

```mermaid
graph LR
	s((s)):::highlight --> a((a))
		s --> b((b))
		a --> G((G)):::highlight
		b --> G
		a --> b
		b --> a

	classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
```

- **State Space**: 4 nodes (S, a, b, G)
- **Search Tree**: Could expand infinitely 

```mermaid
graph TD
		s:::highlight --> a1[a]
		s --> b1[b]

		a1 --> b2[b]
		a1 --> G1[G]:::highlight

		b1 --> a2[a]
		b1 --> G2[G]:::highlight

		b2 --> a3[...]
		a2 --> b3[...]

		%% (Expansion could continue infinitely because of the cycle a <-> b)
		classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
```

This is why search algorithms must track **visited states** to avoid infinite loops!

### 2.1.3 Search Problem Examples

#### Example 1: Pacman

**Problem**: Navigate Pacman through a maze to collect all food pellets.

<img src="images/ch2/pacman.png" alt="Pacman"/>

**State Space Design:**
- **State**: Pacman's current position + remaining food pellets
- **Successor Function**: Moving in one of four cardinal directions (N, E, S, W)
- **Start State**: Pacman at initial position with all food pellets present
- **Goal Test**: All food pellets have been collected

```{mermaid}
graph TD
    A[Pacman Position + Food] -->|Move N| B[New Position + Food]
    A -->|Move E| C[New Position + Food]
    A -->|Move S| D[New Position + Food]
    A -->|Move W| E[New Position + Food]
    style A fill:#FFFF00
```

#### Example 2: Traveling in Romania

**Problem**: Find the shortest route between cities.

<img src="images/ch2/romania.png" alt="Romania Map"/>

**State Space Design:**
- **State**: Current city location
- **Successor Function**: Moving from one city to another connected by a road (cost = distance)
- **Start State**: Initial city (e.g., Arad)
- **Goal Test**: Reaching destination city (e.g., Bucharest)

```{mermaid}
graph LR
    Arad -->|140| Sibiu
    Sibiu -->|99| Fagaras
    Fagaras -->|211| Bucharest
    Sibiu -->|80| Rimnicu
    Rimnicu -->|97| Pitesti
    Pitesti -->|101| Bucharest
    style Arad fill:#90EE90
    style Bucharest fill:#FFB6C1
```

#### Example 3: Eight-Puzzle

**Problem**: The Eight-Puzzle is a sliding puzzle that consists of a 3x3 grid with eight numbered tiles and one empty space. The objective is to arrange the tiles in a specific order by sliding them into the empty space.

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20240726131204/8-puzzle-Problem.webp" alt="Eight Puzzle" width="300"/>
<img src="https://media.geeksforgeeks.org/wp-content/uploads/puzzle-1.jpg" alt="Eight Puzzle" width="300"/>

**State Space Design:**
- **State**: Configuration of tiles on the 3×3 board
- **Successor Function**: Sliding a tile into the empty space
- **Start State**: Initial scrambled configuration
- **Goal Test**: Tiles arranged in desired order (1-8 with blank in corner)

#### Example 4: Online Maze Search

<img src="images/ch2/maze.png" alt="Maze"/>

**State Space Design:**
- **State**: Agent's current position in the maze
- **Successor Function**: Moving in one of four cardinal directions (up, down, left, right) based on current position and maze layout
- **Start State**: Initial position in the maze
- **Goal Test**: Reaching the exit of the maze

```{note}
Each search problem requires careful consideration of:
- How to represent states efficiently
- What actions are available
- How to test for goal states
- What costs are associated with actions
```

## 2.2 Uninformed Search Algorithms

**Uninformed search** algorithms, also known as **blind search** algorithms, are a category of search strategies that operate without any domain-specific knowledge about the problem being solved. They explore the search space systematically, relying solely on the structure of the state space and the goal test to find a solution.

### Generic Search Framework

All uninformed search algorithms follow this template:

```
Algorithm: GenericSearch(Initial State s, Goal Condition G)
Input: Initial state s, Goal condition G
Output: Path from s to goal, or failure

begin
    LIST ← {s}              // Frontier nodes to explore
    VISIT ← ∅               // Visited nodes
    pred ← empty array      // Predecessor tracking
    
    repeat
        if LIST is empty then
            return failure
        
        // Selection strategy differs by algorithm
        Select node i from LIST
        Delete i from LIST
        Add i to VISIT
        
        if i satisfies G then
            return reconstruct_path(pred, s, i)
        
        // Expand node i
        for each node j adjacent to i do
            if j not in VISIT and j not in LIST then
                Add j to LIST
                pred[j] ← i
    until LIST is empty or goal found
end
```

**Key Components:**

- **LIST**: Frontier nodes (boundary of explored region)
- **VISIT**: Already explored nodes
- **pred**: Predecessor array to reconstruct path
- **Selection Strategy**: Determines which algorithm we get

```{mermaid}
graph TD
    A[Start with initial state in LIST] --> B[Select node from LIST]
    B --> C{Is LIST empty?}
    C -->|Yes| D[Return failure]
    C -->|No| E{Is node a goal?}
    E -->|Yes| F[Return path]
    E -->|No| G[Expand node: add neighbors to LIST]
    G --> H[Mark node as visited]
    H --> B
    style A fill:#90EE90
    style F fill:#FFB6C1
```

### 2.2.1 Breadth-First Search (BFS)

BFS explores nodes level by level, visiting all nodes at depth d before any node at depth d+1.

**Selection Strategy**: FIFO queue (First-In-First-Out)

```{important}
The insertion operation is performed at the **end** of the list, while the deletion operation is performed at the **front** of the list, making it a First-In-First-Out (FIFO) structure. This means nodes are expanded in the order they were added, leading to a **level-by-level exploration** of the search space.
```

**How it works:**
1. Start with initial state in queue
2. Remove first node from queue
3. Check if it's a goal
4. Add all unvisited neighbors to end of queue
5. Repeat

**BFS Exploration Pattern:**

```mermaid

graph LR

	S((S)):::highlight --> a((a))
	S --> d((d))
	S --> p((p))

	a --> b((b))
	a --> c((c))
	d --> b
	d --> c
	d --> e((e))
	c --> e

	e --> f((f))
	f --> G((G)):::highlight
	e --> h((h))
	e --> r((r))

	p --> q((q))
	q --> h
	p --> e

	classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,a,d,p,b,c,e,q,f,h,r explored;
```

```mermaid
graph TD
		S:::highlight --> a
		S --> d
		S --> p

		a --> b
		a --> c

		d --> b2[b]
		d --> c2[c]
		d --> e1[e]

		c --> e2[e]

		p --> q
		p --> e3[e]

		q --> h1[h]

		e1 --> f1[f]
		e1 --> h2[h]
		e1 --> r1[r]

		e2 --> f2[f]
		e2 --> h3[h]
		e2 --> r2[r]

		e3 --> f3[f]
		e3 --> h4[h]
		e3 --> r3[r]

		f1 --> G1[G]:::highlight
		f2 --> G2[G]:::highlight
		f3 --> G3[G]:::highlight
		classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
		classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
		class S,a,d,p,b,c,b2,c2,e1,e2,q,e3,h1,f1,h2,r1,f2,f3,h3,r2,h4,r3 explored;
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| **Complete** | Yes | Will find solution if one exists |
| **Optimal** | Yes* | Finds shallowest (shortest path) goal |
| **Time Complexity** | O(b^d) | Exponential in depth |
| **Space Complexity** | O(b^d) | Must store entire frontier |

*Only when all actions have equal cost

Where:  
- **b**: branching factor (average number of successors)
- **d**: depth of shallowest goal

**Advantages:**
- Guaranteed to find solution
- Finds shortest path (minimum edges)

**Disadvantages:**
- **Memory intensive**: Must store all nodes at current level
- Slow for deep goals

### 2.2.2 Depth-First Search (DFS)

DFS explores as deep as possible along each branch before backtracking.

**Selection Strategy**: LIFO stack (Last-In-First-Out)

```{important}
The insertion and deletion operations in the LIST are performed at the **end** of the list, making it a Last-In-First-Out (LIFO) structure. This means the **most recently added node is expanded first**, leading to a deep exploration of the search space before backtracking.
```

**How it works:**
1. Start with initial state on stack
2. Pop top node from stack
3. Check if it's a goal
4. Push all unvisited neighbors onto stack
5. Repeat

**DFS Exploration Pattern:**

```mermaid

graph LR

	S((S)):::highlight --> a((a))
	S --> d((d))
	S --> p((p))

	a --> b((b))
	a --> c((c))
	d --> b
	d --> c
	d --> e((e))
	c --> e

	e --> f((f))
	f --> G((G)):::highlight
	e --> h((h))
	e --> r((r))

	p --> q((q))
	q --> h
	p --> e

	classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,a,b,c,e,f explored;
```

```mermaid
graph TD
		S:::highlight --> a
		S --> d
		S --> p

		a --> b
		a --> c

		d --> b2[b]
		d --> c2[c]
		d --> e1[e]

		c --> e2[e]

		p --> q
		p --> e3[e]

		q --> h1[h]

		e1 --> f1[f]
		e1 --> h2[h]
		e1 --> r1[r]

		e2 --> f2[f]
		e2 --> h3[h]
		e2 --> r2[r]

		e3 --> f3[f]
		e3 --> h4[h]
		e3 --> r3[r]

		f1 --> G1[G]:::highlight
		f2 --> G2[G]:::highlight
		f3 --> G3[G]:::highlight
		classDef highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
		classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
		class S,a,b,c,e2,f2 explored;
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| **Complete** | No | Can get stuck in infinite paths |
| **Optimal** | No | May find longer paths first |
| **Time Complexity** | O(b^m) | Can explore entire tree |
| **Space Complexity** | O(bm) | Only stores single path |

Where:  
- **m**: maximum depth of any node

**Advantages:**
- **Memory efficient**: Only stores nodes on current path
- Fast if solutions are deep

**Disadvantages:**
- Can get stuck in infinite loops (needs cycle detection)
- May find suboptimal solutions
- Not complete in infinite spaces


### Comparison: BFS vs DFS

<img src="images/ch2/bfsdfs.png" alt="BFS vs DFS" style="width: 100%;" />

| Aspect | BFS | DFS |
|--------|-----|-----|
| **Frontier Shape** | Wide, shallow | Deep, narrow |
| **Memory** | O(b^d) - HIGH | O(bm) - LOW |
| **Optimal** | Yes (equal costs) | No |
| **Complete** | Yes | No (with cycles) |
| **Best For** | Shallow goals | Deep solutions, memory limited |

### 2.2.3 Uniform-Cost Search (UCS)

UCS generalizes BFS to handle **weighted graphs** where actions have different costs. It is an uninformed search algorithm that expands the least costly node first, ensuring that the path to the goal is optimal in terms of cost. It uses a priority queue to manage the frontier, where nodes are prioritized based on their cumulative cost from the start state.

**Selection Strategy**: Priority queue ordered by path cost g(n)

**Key Idea**: Always expand the node with the **lowest cumulative path cost** from start.

**Path Cost**: g(n) = sum of edge costs from start to node n

**How it works:**
1. Maintain priority queue of nodes, ordered by g(n)
2. Always expand node with smallest g(n)
3. Update costs when shorter paths are found

**Detailed Example:**

```mermaid
graph LR

    S((S)) -->|12| a((a))
    S -->|6| d((d))
    S -->|1| p((p))   

    a -->|2| b((b))
    a -->|3| c((c))
    d -->|4| b
    d -->|6| c
    d -->|8| e((e))
    c -->|2| e

    e -->|2| f((f))
    f -->|1| G((G))
    e -->|5| h((h))
    e -->|9| r((r))

    p -->|1| q((q))
    q -->|2| h
    p -->|2| e
```

**After UCS execution:**

```mermaid
graph LR

    S((S)) -->|12| a((a))
    S -->|6| d((d))
    S -->|1| p((p))

    a -->|2| b((b))
    a -->|3| c((c))
    d -->|4| b
    d -->|6| c
    d -->|8| e((e))
    c -->|2| e

    e -->|2| f((f))
    f -->|1| G((G))
    e -->|5| h((h))
    e -->|9| r((r))

    p -->|1| q((q))
    q -->|2| h
    p -->|2| e

    %% Highlight UCS order (by cost)
    style S fill:#ffcc00,stroke:#333,stroke-width:2px
    style p fill:#ffe599,stroke:#333,stroke-width:2px
    style q fill:#f9cb9c,stroke:#333,stroke-width:2px
    style e fill:#9fc5e8,stroke:#333,stroke-width:2px
    style f fill:#6fa8dc,stroke:#333,stroke-width:2px
    style G fill:#38761d,stroke:#fff,stroke-width:2px
```

**Properties:**

| Property | Value |
|----------|-------|
| **Complete** | Yes |
| **Optimal** | Yes |
| **Time Complexity** | O(b^(C*/ε)) |
| **Space Complexity** | O(b^(C*/ε)) |

Where:  
- **C***: cost of optimal solution
- **ε**: minimum edge cost

```{note}
UCS is identical to BFS when all edge costs are equal.
```

### 2.2.5 Bidirectional Search

Run two simultaneous searches: one forward from start, one backward from goal. Stop when they meet.

**Key Idea**: Two small searches are faster than one large search.

```python
Algorithm BiSearch(Initial State: s, Goal State: g)
 begin
	FLIST= { s }; BLIST= { g };
	repeat
		Select current node if from FLIST based on pre-defined strategy;
		Select current node ib from BLIST based on pre-defined strategy;
		Delete nodes if and ib respectively from FLIST and BLIST;
		Add nodes if and ib to the hash tables FVISIT and BVISIT, respectively;
		for each node j ∈ A(if) not in FVISIT reachable from if do
			Add j to FLIST; pred(j)=if;
		for each node j ∈ B(ib) not in BVISIT do
			Add j to BLIST; succ(j)=ib;
	until FLIST or BLIST is empty or (FLIST ∩ BLIST)= {};
	if (either FLIST or BLIST is empty) return failure;
		else return success;
	{ Reconstruct source-goal path by selecting any node in FLIST ∩ BLIST
	and tracing predecessor and successor lists from that node; }
 end
```

**Complexity Improvement:**
- Normal BFS: O(b^d)
- Bidirectional: O(2 × b^(d/2)) = O(b^(d/2))

**Example:** If b=10, d=6:
- Normal: 10^6 = 1,000,000 nodes
- Bidirectional: 2 × 10^3 = 2,000 nodes

```{mermaid}
graph LR
    S[Start] -->|Forward| M[Meet in middle]
    G[Goal] -->|Backward| M
    style S fill:#90EE90
    style G fill:#FFB6C1
    style M fill:#FFD700
```

**Requirements:**
- Must be able to generate predecessors
- Must be able to check if nodes from both searches match

**Challenges:**
- Requires goal state to be explicitly known
- Need to detect when searches meet
- More complex implementation

### Summary: Uninformed Search Algorithms

| Algorithm | Complete | Optimal | Time | Space | When to Use |
|-----------|----------|---------|------|-------|-------------|
| **BFS** | Yes | Yes* | O(b^d) | O(b^d) | Shallow goals, all costs equal |
| **DFS** | No | No | O(b^m) | O(bm) | Memory limited, deep solutions |
| **UCS** | Yes | Yes | O(b^(C*/ε)) | O(b^(C*/ε)) | Weighted graphs |
| **Bidirectional** | Yes | Yes* | O(b^(d/2)) | O(b^(d/2)) | Know both start and goal |

*Optimal only for equal-cost actions

```{important}
**Key Takeaway**: Uninformed search is inefficient for large state spaces. We need **informed search** that uses problem-specific knowledge!
```

## 2.3 Informed Search Algorithms

**Informed search** uses problem-specific knowledge through **heuristic functions** to guide search more efficiently toward goals.

### Heuristic Function h(n)

**Definition**: Estimated cost from node n to the nearest goal.

**Example** (8-puzzle): 
- h(n) = number of misplaced tiles
- h(n) = sum of Manhattan distances of tiles

**Key Properties:**

1. **Admissible**: Never overestimates true cost  
   h(n) ≤ h*(n) where h*(n) is true cost to goal

2. **Consistent** (Monotonic): Satisfies triangle inequality  
   h(n) ≤ c(n,n') + h(n') for every edge (n,n')

```{note}
Consistency implies admissibility, but not vice versa.
```

### Evaluation Functions

**g(n)**: Actual cost from start to n  
**h(n)**: Estimated cost from n to goal  
**f(n)**: Total estimated cost of solution through n

### 2.3.1 Best-First Search (Greedy Search)

**Evaluation Function**: f(n) = h(n)

**Strategy**: Expand node that **appears** closest to goal based on heuristic estimate.

```{important}
Best-First Search selects the most promising node to expand based on a **heuristic function**, which estimates the cost from the current node to the goal. Unlike UCS which uses actual path cost, Best-First uses only the estimated remaining cost.
```

**How it works:**
1. Maintain priority queue ordered by h(n)
2. Always expand node with smallest h(n)
3. Ignore cost to reach current node

### Best-First Search (Greedy Search) VS The Uniform Cost Search

**Using UCS** (actual costs only):

```mermaid
graph LR
    S -->|1| a
    a -->|3| d
    a -->|1| b
    a -- 8 --> e
    b -->|1| c
    e -->|1| d
    d -->|2| G
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G Highlight;
	class a,b,c explored;
```
The node c is a dead end. The next node with the lowest cost is d (3), so it is selected for expansion next.

```mermaid
graph LR
    S -->|1| a
    a -->|3| d
    a -->|1| b
    a -- 8 --> e
    b -->|1| c
    e -->|1| d
    d -->|2| G
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G Highlight;
	class a,d,b,c explored;
```


**Using Best-First Search** (heuristic only): an estimation of the cost from each node to the goal node G is provided by a heuristic function h(n).

```mermaid
graph LR
    S(S<br>h=6) -->|1| a(a<br>h=5)
    a -->|3| d(d<br>h=2)
    a -->|1| b(b<br>h=6)
    a -- 8 --> e(e<br>h=1)
    b -->|1| c(c<br>h=7)
    e -->|1| d
    d -->|2| G(G<br>h=0)
```
A Heuristic Function h(n) is used to estimate the cost from node n to the goal node G. It could be:  

- The straight-line distance between two cities in a route-finding problem.	
- The Manhattan distance in a grid-based pathfinding problem.
- The number of misplaced tiles in the 8-puzzle problem.

<div style="display: flex; gap: 1.5rem; align-items: center;">
	<img src="images/ch2/Best1.png" alt="Best-First Search" style="width: 300px; border: 1px solid #ccc; border-radius: 8px;">
    <img src="images/ch2/Best2.svg" alt="Best-First Search" style="width: 300px; border: 1px solid #ccc; border-radius: 8px;">
    <img src="images/ch2/Best3.png" alt="Best-First Search" style="width: 300px; border: 1px solid #ccc; border-radius: 8px;">
</div>

```mermaid
graph LR
    S(S<br>h=6) -->|1| a(a<br>h=5)
    a -->|3| d(d<br>h=2)
    a -->|1| b(b<br>h=6)
    a -- 8 --> e(e<br>h=1)
    b -->|1| c(c<br>h=7)
    e -->|1| d
    d -->|2| G(G<br>h=0)
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G Highlight;
	class a,e,d explored;
```

```mermaid
graph LR
	S(S<br>h=6):::Highlight -->|1| a(a<br>h=5)
	a -->|3| d(d<br>h=2)
	a -->|1| b(b<br>h=6)
	a -- 8 --> e(e<br>h=1)
	b -->|1| c(c<br>h=7)
	e -->|1| d1[d<br>h=2]
	d1 -->|2| G(G<br>h=0)
	d -->|2| G1[G<br>h=0]
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G,G1 Highlight;
	class a,e,d1 explored;
```

### 2.3.2 A* Search

A* is the **most widely used** informed search algorithm, combining actual and estimated costs.

**Evaluation Function**:

$$f(n) = g(n) + h(n)$$

Where:
- **g(n)**: Actual cost from start to n
- **h(n)**: Estimated cost from n to goal
- **f(n)**: Estimated total cost of solution through n

```python
Algorithm GenericSearch(Initial State: s, Goal Condition: G)
 begin
	LIST= { s };
	repeat
		Select current node i from LIST based on pre-defined strategy;
		Delete node i from LIST;
		Add node i to the hash table VISIT;
		for all nodes j ∈ A(i) directly connected to i via transition do
		begin
			if (j is not in VISIT) add j to LIST;
			pred(j)=i;
			choose the node j according to f(j):
				f(j) = min{c(j)} // Uniform Cost Search
				f(j) = min{h(j)} // Best-First Search
				f(j) = min{c(j) + h(j)} // A* Search 
		end
	until LIST is empty or current node i satisfies G;
	if current node satisfies G return success
		else return failure;
	{ The predecessor array can be used to trace back path from i to s }
 end
```

```mermaid
graph LR
    S(S<br>h=6) -->|1| a(a<br>h=5)
    a -->|3| d(d<br>h=2)
    a -->|1| b(b<br>h=6)
    a -- 8 --> e(e<br>h=1)
    b -->|1| c(c<br>h=7)
    e -->|1| d
    d -->|2| G(G<br>h=0)
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G Highlight;
	class a,d explored;
```

**How it works:**
1. Maintain priority queue ordered by f(n) = g(n) + h(n)
2. Always expand node with smallest f(n)
3. Balance between:
   - Nodes close to start (small g)
   - Nodes close to goal (small h)

```mermaid
graph LR
	S(S<br>f=0+6):::Highlight -->|1| a(a<br>f=1+5)
	a -->|3| d(d<br>f=4+2)
	a -->|1| b(b<br>f=2+6)
	a -- 8 --> e(e<br>f=9+1)
	d -->|2| G1[G<br>f=6+0]
	classDef Highlight fill:#ffeb3b,stroke:#fbc02d,stroke-width:3px;
	classDef explored fill:#c8e6c9,stroke:#388e3c,stroke-width:2px;
	class S,G,G1 Highlight;
	class a,d explored;
```


### Why A* is Optimal

**Theorem**: If h(n) is admissible, A* finds the optimal solution.

**Proof Sketch**:

1. Suppose A* returns suboptimal goal G₂ with cost C₂
2. Let G be optimal goal with cost C* < C₂
3. Some unexpanded node n must be on optimal path to G
4. Since h is admissible:
   - f(n) = g(n) + h(n) ≤ g(n) + h*(n) = C*
5. Therefore: f(n) ≤ C* < C₂ = f(G₂)
6. But A* expands nodes in order of f-value
7. So n would be expanded before G₂
8. Contradiction! ∎

```{important}
A* is **optimally efficient**: No other optimal algorithm with same heuristic expands fewer nodes.
```

### When Should A* Terminate?

A* should terminate **when the goal node is selected for expansion from the list**. This ensures the path found is optimal.

**Example demonstrating why:**

```{mermaid}
graph LR
    S["S (h=3)"] -->|2| a["a (h=2)"]
    S -->|2| b["b (h=1)"]
    a -->|2| G1["G"]
    b -->|3| G2["G"]
    style S fill:#90EE90
    style G1 fill:#FFB6C1
    style G2 fill:#FFB6C1
```

| Step | Node | f = g + h | Result |
|------|------|-----------|--------|
| 1 | S | 0 + 3 = 3 | Expand, add a(f=4), b(f=3) |
| 2 | b | 2 + 1 = 3 | Expand, add G(f=5) via b |
| 3 | a | 2 + 2 = 4 | Expand, add G(f=4) via a |
| 4 | G | 4 + 0 = 4 | **Goal selected for expansion → terminate!** |

Optimal path: S → a → G (cost = 4), not S → b → G (cost = 5)

```{warning}
If A* terminates when the goal is **first generated** (not selected for expansion), it may return a suboptimal path!
```

### Is A* Always Optimal?

A* is optimal **only if** the heuristic is **admissible**:

**Admissible**: h(n) never overestimates the true cost to reach the goal.  
For every node n: h(n) ≤ h*(n) where h*(n) is the actual cost to goal.

**Counter-example with inadmissible heuristic:**

```{mermaid}
graph LR
    S["S (h=7)"] -->|1| a["a (h=6)"]
    S -->|5| G1["G (h=0)"]
    a -->|3| G2["G (h=0)"]
    style S fill:#90EE90
    style G1 fill:#FFB6C1
    style G2 fill:#FFB6C1
```

- h(S) = 7 (but true cost is 4 via S→a→G) — **Overestimates!**
- A* explores: S(f=7) → a(f=7) → but G via S has f=5, so A* might return cost=5 instead of optimal cost=4

```{important}
**If h(n) is admissible, A* will always find the optimal path to the goal when it terminates.**
```

### 2.3.3 Designing Heuristics

Good heuristics make search efficient. Here are examples for common problems.

### Example 1: 8-Puzzle

**Problem**: Slide tiles to reach goal configuration.

**Heuristic 1: Misplaced Tiles**

$$h_1(n) = \text{number of tiles in wrong position}$$

- **Admissible?** Yes (each misplaced tile requires ≥1 move)
- **Quality**: Moderate

**Heuristic 2: Manhattan Distance**

$$h_2(n) = \sum_{\text{tiles}} |x_{current} - x_{goal}| + |y_{current} - y_{goal}|$$

- **Admissible?** Yes (tiles can't jump, need ≥ Manhattan distance moves)
- **Quality**: Better than h₁

**Example State:**
```
Current:        Goal:
1 2 3           1 2 3
4 5 6           4 5 6
7   8           7 8

h₁ = 1 (tile 8 misplaced)
h₂ = 1 (tile 8 needs 1 move)
```

**Empirical Results:**

| Heuristic | Avg Nodes Expanded |
|-----------|-------------------|
| None (UCS) | ~170,000 |
| Misplaced (h₁) | ~500 |
| Manhattan (h₂) | ~50 |

```{note}
**Dominance**: If h₂(n) ≥ h₁(n) for all n, then h₂ **dominates** h₁.  
A* with h₂ never expands more nodes than with h₁.
```

### Techniques for Creating Heuristics

**1. Relaxed Problem**

Remove constraints from original problem:
- 8-puzzle: Allow tiles to move anywhere → Manhattan distance
- Route planning: Ignore roads → straight-line distance

**2. Pattern Databases**

Precompute optimal costs for subproblems:
- 8-puzzle: Store costs for positioning each tile
- Rubik's cube: Store costs for corner positions

**3. Learning**

Train a function to estimate costs:
- Use neural network
- Learn from experience

### Properties of Good Heuristics

✓ **Admissible**: Guarantees optimality  
✓ **Consistent**: Enables efficient search  
✓ **Accurate**: Close to true cost (but never over)  
✓ **Fast to compute**: Overhead shouldn't exceed savings

```{tip}
**Trade-off**: More accurate heuristics expand fewer nodes but may take longer to compute. Find the sweet spot!
```

## 2.4 Local Search Algorithms

**Local search** algorithms focus on exploring the immediate neighborhood of the current solution to find an improved solution. Unlike global search algorithms that explore the entire search space:

- They don't track paths taken
- They only care about **final state quality**
- They use **state-centric loss functions**
- They make incremental changes to iteratively refine solutions

```{important}
**When is Local Search Useful?**

Local search is ideal for scenarios where the **final state matters** rather than the path to reach it:
- **Eight-queens problem**: The final arrangement matters, not the move sequence
- **Traveling Salesperson Problem**: The final route matters, not the order cities were explored
- **Scheduling problems**: The final schedule matters, not how it was constructed
```

**Applications:**
- Scheduling problems
- Circuit design
- Eight-queens (complete-state approach)
- Traveling salesperson problem
- VLSI layout
- Vehicle routing

```{mermaid}
graph TD
    A[Current State] --> B{Evaluate Neighbors}
    B --> C[Choose Better Neighbor]
    C --> D{Better than current?}
    D -->|Yes| A
    D -->|No| E[Stop - Local Optimum]
    style A fill:#FFD700
    style E fill:#FFB6C1
```

### 2.4.1 Hill Climbing

**Hill climbing** is a greedy local search algorithm that iteratively moves towards the neighbor state with the lowest loss (or highest value) until no better neighbors are found.

**Algorithm:**
```
Algorithm: HillClimbing(Initial State s, Loss Function L)
begin
    current ← s
    loop
        neighbors ← generate_neighbors(current)
        if neighbors is empty then
            return current
        
        next ← neighbor with minimum L(neighbor)
        
        if L(next) ≥ L(current) then
            return current  // Local optimum
        
        current ← next
end
```

### Hill Climbing Example 1: Eight-Queens

Solving the 8-queens problem using Hill Climbing:
1. **Start** with a random configuration of 8 queens on the chessboard
2. **Evaluate** the number of pairs of queens attacking each other (the loss function)
3. **Move** a queen to a different position in its column to reduce attacking pairs
4. **Repeat** until zero attacking pairs (success) or no improvement possible (stuck)

- **State**: 8 queens on board (may attack each other)
- **Neighbor**: Move one queen to different row in same column
- **Loss**: Number of pairs of queens attacking each other
- **Goal**: Loss = 0

### Hill Climbing Example 2: Traveling Salesperson

Solving TSP using Hill Climbing:
1. **Start** with a random tour of all cities
2. **Evaluate** the total distance of the tour (loss function)
3. **Swap** positions of two cities in the tour to reduce total distance
4. **Repeat** until no swap improves the tour

### The Problem of Local Optima

```{warning}
**Hill Climbing can get stuck** in local optima, where no neighboring state has a better value, but the current state is not the global optimum. This can lead to suboptimal solutions!
```

<img src="images/ch2/hill3.svg" alt="Local Optima" style="width: 100%; border: 1px solid #ccc; border-radius: 8px;" />

**Problems with Hill Climbing:**

```{mermaid}
graph LR
    A[Local Maximum] -.-> B[Better]
    C[Plateau] -.-> B
    D[Ridge] -.-> B
    B --> E[Global Optimum]
    style A fill:#FFB6C1
    style E fill:#90EE90
```

1. **Local Maxima**: All neighbors worse, but not globally optimal
2. **Plateaus**: Flat regions where all neighbors have same value
3. **Ridges**: Sequence of local maxima difficult to navigate

**Solutions:**
- **Random restart**: Try multiple random initial states
- **Stochastic hill climbing**: Choose randomly among uphill moves
- **First-choice**: Take first improvement found
- **Random walk**: Sometimes take random steps

### 2.4.2 Simulated Annealing

**Simulated annealing** is a probabilistic optimization algorithm inspired by the annealing process in metallurgy. It allows occasional **uphill moves** (accepting worse solutions) to escape local optima.

**Key Ideas:**
- Accept moves to worse solutions with a certain probability
- This enables the search to escape local optima and explore more thoroughly
- Since worse solutions can be accepted, the algorithm stores the **best solution found** during search
- The probability of accepting worse solutions **decreases over time** (controlled by temperature)

**Algorithm:**
```
Algorithm: SimulatedAnnealing(Initial State s, Loss L, Schedule T(t))
begin
    current ← s
    best ← s
    T ← T₀  // Initial temperature (same magnitude as typical loss differences)
    t ← 1
    
    loop
        next ← random neighbor of current
        ΔL ← L(next) - L(current)
        
        if ΔL < 0 then
            current ← next  // Always accept improvement
        else
            // Accept with probability exp(-ΔL/T)
            with probability exp(-ΔL/T) do
                current ← next
        
        if L(current) < L(best) then
            best ← current
        
        t ← t + 1
        T ← T₀ / log(t + 1)  // Cool down
        
    until no improvement in best for N iterations
    return best
end
```

**Acceptance Probability:**

$$P(\text{accept}) = \begin{cases}
1 & \text{if } \Delta L < 0 \text{ (improvement)} \\
e^{-\Delta L / T} & \text{if } \Delta L \geq 0 \text{ (worse move)}
\end{cases}$$

**Temperature Schedule:**

- **High T**: Accept bad moves frequently (explore widely)
- **Low T**: Accept bad moves rarely (exploit current region)
- **T → 0**: Becomes hill climbing

**Common Schedules:**
- Linear: T(t) = T₀ - αt
- Exponential: T(t) = T₀ × γᵗ (γ < 1)
- Logarithmic: T(t) = T₀ / log(t)

**Properties:**

✓ Can escape local optima  
✓ Guaranteed to find global optimum (with slow enough cooling)  
✓ Simple to implement  
✗ Requires tuning temperature schedule  
✗ May be slow to converge

### 2.4.3 Tabu Search

**Tabu search** is an advanced local search algorithm that enhances hill climbing by incorporating memory structures to avoid cycling back to previously visited solutions.

**Key Idea**: Maintain a list of "tabu" moves or solutions that are temporarily forbidden, allowing the search to explore new areas of the solution space and escape local optima.

**Algorithm:**
```
Algorithm: TabuSearch(Initial State s, Loss L, Tabu Size k)
begin
    current ← s
    best ← s
    tabu_list ← empty list  // The tabu list
    
    loop
        Add current to tabu_list
        
        // Find best unvisited neighbor
        if all neighbors of current are in tabu_list then
            break  // No valid moves
        
        next ← best neighbor not in tabu_list
        
        // May also accept tabu moves if they're better than best ever (aspiration)
        current ← next
        
        if L(current) < L(best) then
            best ← current
    
    return best
end
```

**Enhanced Tabu Search Strategies:**

1. **Tabu List Size**: Limit to fixed number of recent moves. If exceeded, remove oldest entry
2. **Aspiration Criteria**: Allow tabu moves if they result in solution better than any previously found
3. **Intensification**: Focus search on promising areas when good solutions are found
4. **Diversification**: Periodically jump to unexplored regions to explore new areas

**Features:**

- **Tabu List**: Fixed-size memory of recent moves/states
- **Aspiration Criteria**: Override tabu if solution is best so far
- **Diversification**: Periodically jump to unexplored regions
- **Intensification**: Focus search on promising regions

**Advantages:**
- Escapes local optima by forbidding backtracking
- Often finds better solutions than hill climbing
- Balances exploration and exploitation

**Disadvantages:**
- Requires tuning tabu list size
- More complex than simpler methods

```{note}
Tabu Search is particularly effective for combinatorial optimization problems such as the traveling salesman problem, job scheduling, and vehicle routing, where the search space is large and complex.
```

### Comparison: Local Search Methods

| Method | Escape Local Optima? | Memory | Best For |
|--------|---------------------|--------|----------|
| **Hill Climbing** | No | None | Simple problems, quick solutions |
| **Random Restart** | Yes | None | Many local optima |
| **Simulated Annealing** | Yes (probabilistic) | None | Large state spaces |
| **Tabu Search** | Yes (deterministic) | Tabu list | Avoiding cycles, structured problems |
| **Genetic Algorithm** | Yes | Population | Complex optimization |

## 2.5 Genetic Algorithms

**Genetic algorithms** borrow their paradigm from the process of biological evolution, working with a population of solutions whose fitness is improved over time.

### Core Concepts

- **Chromosome**: Solution encoded as a string (often bits, but can be other representations)
- **Population**: Set of candidate solutions
- **Fitness**: Quality measure of each solution
- **Selection**: Choose parents based on fitness (fitter individuals more likely to reproduce)
- **Crossover**: Combine parents to create offspring by exchanging segments
- **Mutation**: Random changes to maintain diversity

### Genetic Operators Explained

| Operator | Description |
|----------|-------------|
| **Selection** | Fittest individuals are selected from current population to serve as parents for next generation |
| **Crossover** | Pairs of solutions are selected, and their characteristics are recombined by exchanging segments of their chromosomes |
| **Mutation** | Random changes to chromosomes (e.g., flipping bits, swapping elements) to introduce variation |

```{mermaid}
graph TD
    A[Initialize Population] --> B[Evaluate Fitness]
    B --> C[Selection]
    C --> D[Crossover]
    D --> E[Mutation]
    E --> B
    B --> F{Converged?}
    F -->|No| C
    F -->|Yes| G[Best Solution]
    style A fill:#90EE90
    style G fill:#FFB6C1
```

**Algorithm:**
```
Algorithm: GeneticAlgorithm(Population Size N, Generations T)
begin
    population ← random initialization of N chromosomes
    t ← 1
    
    repeat
        // Select parents based on fitness
        population ← Select(population)  // Uses loss function
        
        // Crossover to create offspring (some may remain unchanged)
        population ← Crossover(population)
        
        // Mutate with some probability (mutation rate parameter)
        population ← Mutate(population)
        
        t ← t + 1
    until convergence condition is met
    
    return best chromosome from population
end
```

**Example: TSP with GAs**

- **Chromosome**: Permutation of cities [3,1,4,2,5] representing tour order
- **Fitness**: Negative of tour length (shorter tours = higher fitness)
- **Crossover**: Order crossover - preserve relative city order from parents
- **Mutation**: Swap positions of two cities in the tour

```{note}
For the traveling salesperson problem, a chromosome could be a sequence of city indices defining the tour. The "fitness" is defined by the cost (negative of tour length) of the corresponding tour.
```